# LLM GMC — Name That LLM baseline

This competition-linked notebook computes a reproducible local validation score. It does not submit predictions to the leaderboard.

Baseline: word + character TF-IDF features with class-balanced LinearSVC. If `prompt_id` is present, validation is grouped by prompt to reduce prompt leakage.

In [ ]:
from pathlib import Path
import json
import time
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import FeatureUnion, Pipeline
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, balanced_accuracy_score, classification_report

DATA_DIR = Path('/kaggle/input/competitions/name-that-llm')
print('Dataset directory exists:', DATA_DIR.exists())
print('Dataset files:', sorted(p.name for p in DATA_DIR.glob('*')) if DATA_DIR.exists() else [])
assert (DATA_DIR / 'train.csv').exists(), f'train.csv not found under {DATA_DIR}'
train = pd.read_csv(DATA_DIR / 'train.csv')
print('Train shape:', train.shape)
print('Columns:', train.columns.tolist())
assert {'text', 'model'}.issubset(train.columns), 'Expected text and model columns'
train['text'] = train['text'].fillna('').astype(str)
train['model'] = train['model'].astype(str)
print('Model classes:', train['model'].nunique())
display(train['model'].value_counts())

In [ ]:
if 'prompt_id' in train.columns and train['prompt_id'].notna().any():
    groups = train['prompt_id'].fillna('__missing_prompt_id__').astype(str)
    splitter = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
    train_idx, valid_idx = next(splitter.split(train, groups=groups))
    split_method = 'GroupShuffleSplit by prompt_id'
else:
    train_idx, valid_idx = train_test_split(range(len(train)), test_size=0.20, random_state=42, stratify=train['model'])
    split_method = 'stratified random split (prompt_id unavailable)'

train_part = train.iloc[list(train_idx)].copy()
valid_part = train.iloc[list(valid_idx)].copy()
print('Validation method:', split_method)
print('Training rows:', len(train_part))
print('Validation rows:', len(valid_part))
print('Validation classes:', valid_part['model'].nunique())

In [ ]:
import re
import numpy as np
from scipy.sparse import hstack, csr_matrix
from sklearn.model_selection import GroupKFold
from sklearn.metrics import balanced_accuracy_score

# Same-prompt pairwise attribution diagnostic.
# This tests whether the difficult version pairs have transferable fingerprints
# after prompt content is controlled.
texts = train['text'].fillna('').astype(str).to_numpy()
labels = train['model'].astype(str).to_numpy()
groups = train['prompt_id'].fillna('__missing_prompt_id__').astype(str).to_numpy()

pair_groups = [
    ['gemini-3.7-flash', 'gemini-3.8-flash'],
    ['gemini-3.1-pro-preview', 'gemini-3.6-flash', 'gemini-3.7-flash', 'gemini-3.8-flash'],
    ['gpt-5.6-luna', 'gpt-5.6-sol', 'gpt-5.6-terra'],
    ['glm-5.3', 'glm-5.3-flash'],
    ['claude-fable-5', 'claude-opus-4-6', 'claude-opus-4-8', 'claude-opus-5', 'claude-sonnet-5'],
]

def style_one(text):
    words = re.findall(r"\b[\w']+\b", text.lower())
    n_words, n_chars = max(len(words),1), max(len(text),1)
    sentences = max(len(re.findall(r'[.!?]+', text)),1)
    lengths = np.asarray([len(w) for w in words], dtype=float)
    punct = '.,;:!?()[]{}"\'/-'
    return np.asarray([
        np.log1p(len(text)), np.log1p(len(words)), np.log1p(sentences),
        lengths.mean() if len(lengths) else 0., lengths.std() if len(lengths) else 0.,
        len(set(words))/n_words, sum(c.isdigit() for c in text)/n_chars,
        sum(c.isupper() for c in text)/n_chars, sum(c.isspace() for c in text)/n_chars,
        text.count('\\n')/n_chars, text.count('\\n\\n')/n_chars,
        *[text.count(c)/n_chars for c in punct], text.count('#')/n_chars,
        text.count('*')/n_chars, text.count('```')/n_chars,
    ], dtype=np.float32)

style = np.vstack([style_one(x) for x in texts])

# Within-prompt descriptive contrasts: each feature is centered by the prompt mean.
centered = style.copy()
for gid in np.unique(groups):
    mask = groups == gid
    centered[mask] -= centered[mask].mean(axis=0, keepdims=True)

results = []
for pair in pair_groups:
    mask = np.isin(labels, pair)
    idx = np.where(mask)[0]
    if len(np.unique(labels[idx])) < 2 or len(np.unique(groups[idx])) < 5:
        continue
    pair_y = labels[idx]
    pair_g = groups[idx]
    # Prompt-centered stylometry evaluates style after controlling for prompt-level length/topic effects.
    scores = []
    for fold, (tr, va) in enumerate(GroupKFold(n_splits=5).split(idx, pair_y, groups=pair_g), 1):
        model = LinearSVC(C=1.0, class_weight='balanced', max_iter=1500)
        model.fit(centered[idx[tr]], pair_y[tr])
        scores.append(float(balanced_accuracy_score(pair_y[va], model.predict(centered[idx[va]]))))
    # Raw within-prompt feature summaries by model, useful for interpreting the fingerprint.
    summary = {}
    for lab in pair:
        lm = mask & (labels == lab)
        summary[lab] = {
            'n': int(lm.sum()),
            'mean_words': float(np.mean([len(re.findall(r"\b[\w']+\b", x)) for x in texts[lm]])),
            'mean_chars': float(np.mean([len(x) for x in texts[lm]])),
            'mean_sentences': float(np.mean([max(len(re.findall(r'[.!?]+', x)),1) for x in texts[lm]])),
            'mean_punctuation_density': float(np.mean(style[lm, 10:25])),
        }
    results.append({
        'pair': pair,
        'rows': int(len(idx)),
        'prompt_groups': int(len(np.unique(pair_g))),
        'prompt_centered_style_mean': float(np.mean(scores)),
        'prompt_centered_style_std': float(np.std(scores, ddof=1)),
        'fold_scores': scores,
        'model_summary': summary,
    })
    print(f"PAIR {pair}: centered-style BA={np.mean(scores):.4f} +/- {np.std(scores, ddof=1):.4f}")

summary = {
    'experiment': 'same-prompt pairwise version fingerprint diagnostic',
    'split_method': '5-fold GroupKFold by prompt_id',
    'results': results,
    'interpretation': 'Prompt-centered style score estimates how much version-specific signal remains after removing prompt-group average style.'
}
print('\\nSUMMARY:', json.dumps(summary, indent=2))
Path('/kaggle/working/pairwise_results.json').write_text(json.dumps(summary, indent=2))
print('Saved:', '/kaggle/working/pairwise_results.json')


## Same-prompt pairwise version diagnostic

This run controls for prompt content by centering stylometric features within each prompt group, then evaluates difficult model-version groups with GroupKFold.
